# 🚢 China's Import Engine — MDR & HHI Analysis

**Objective:** Identify which product categories and supplier countries expose China's supply chain to the highest risk, using two complementary metrics:

- **Maritime Dependency Ratio (MDR)** — how reliant each product/country is on sea transport
- **Herfindahl-Hirschman Index (HHI)** — how concentrated (and therefore fragile) the supplier base is

**Data:** 2,473,177 trade records · 2016–2021 · Source: UNCTAD

---
## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print('Libraries loaded ✅')

---
## 1. Load & Inspect Data

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────────
# Option A: CSV export from Hyper file
# df = pd.read_csv('../data/china_imports.csv', parse_dates=['Year'])

# Option B: Read directly from Tableau Hyper file
# from tableauhyperapi import HyperProcess, Telemetry, Connection
# ... (see tableauhyperapi docs)

# ── Demo data (remove once you plug in real data) ──────────────────────────
rng = np.random.default_rng(42)
hs_cats = {
    1:  'Animal & Animal Products',  2:  'Vegetable Products',
    3:  'Foodstuffs',                4:  'Mineral Products',
    5:  'Chemicals',                 6:  'Plastics/Rubbers',
    8:  'Wood & Wood Products',      9:  'Textiles',
    12: 'Metals',                    13: 'Machinery/Electrical',
    14: 'Transportation',            15: 'Miscellaneous',
}
countries = ['Russia','USA','Germany','Brazil','Australia',
             'India','South Korea','Japan','France','Saudi Arabia',
             'Indonesia','Chile','Canada','South Africa','Malaysia']
n = 50_000
df = pd.DataFrame({
    'Year':      pd.to_datetime(rng.choice(range(2016, 2022), n).astype(str)),
    'HS':        rng.choice(list(hs_cats.keys()), n),
    'Operation': rng.choice(['import','export'], n, p=[0.65, 0.35]),
    'Area':      rng.choice(countries, n),
    'Region':    rng.choice(['Asia','Europe','Americas','Africa','Oceania'], n),
    'FobValue':  rng.exponential(1e9, n),
    'Tcosts':    rng.exponential(5e7, n),
    'QtyKg':     rng.exponential(1e6, n),
    'Transport': rng.choice(['Sea','Air','Road','Railway'], n,
                             p=[0.70, 0.15, 0.10, 0.05]),
})
df['Description'] = df['HS'].map(hs_cats)
# ──────────────────────────────────────────────────────────────────────────

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Data types & nulls
df.info()

In [ ]:
# Filter to imports only for risk analysis
imports = df[df['Operation'] == 'import'].copy()
print(f'Import records: {len(imports):,}')
print(f'Year range:     {imports["Year"].dt.year.min()} – {imports["Year"].dt.year.max()}')
print(f'HS categories:  {imports["Description"].nunique()}')
print(f'Supplier countries: {imports["Area"].nunique()}')
print(f'Transport modes: {imports["Transport"].unique()}')

---
## 2. Exploratory Overview

In [ ]:
# Transport mode share
mode_share = (imports.groupby('Transport')['FobValue'].sum()
                     .sort_values(ascending=False))
mode_pct = (mode_share / mode_share.sum() * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mode_share.plot(kind='bar', ax=axes[0], color=['#1f77b4','#ff7f0e','#2ca02c','#9467bd'])
axes[0].set_title('Total Import Value by Transport Mode')
axes[0].set_ylabel('FOB Value (€)')
axes[0].tick_params(axis='x', rotation=0)

axes[1].pie(mode_pct, labels=mode_pct.index, autopct='%1.1f%%',
            colors=['#1f77b4','#ff7f0e','#2ca02c','#9467bd'])
axes[1].set_title('Transport Mode Share (%)')

plt.tight_layout()
plt.show()

print('\nSea dominates China\'s import transport — any maritime disruption has outsized impact.')

---
## 3. Maritime Dependency Ratio (MDR)

### What is MDR?

MDR measures what proportion of a product's (or country's) total import value arrives by sea.

$$\text{MDR} = \frac{\sum \text{FobValue}_{\text{Sea}}}{\sum \text{FobValue}_{\text{All modes}}}$$

A high MDR means that if global sea routes are disrupted (e.g. Suez Canal blockage, geopolitical conflict), that supply chain is highly vulnerable.

In [ ]:
def compute_mdr(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    """Compute Maritime Dependency Ratio per group."""
    sea_val   = (df[df['Transport'] == 'Sea']
                   .groupby(group_col)['FobValue'].sum())
    total_val = df.groupby(group_col)['FobValue'].sum()
    mdr = (sea_val / total_val).rename('MDR').reset_index()
    mdr['MDR_pct'] = (mdr['MDR'] * 100).round(1)
    mdr['Risk'] = pd.cut(mdr['MDR_pct'],
                         bins=[0, 40, 65, 100],
                         labels=['Low', 'Moderate', 'High'])
    return mdr.sort_values('MDR_pct', ascending=False)


mdr_product = compute_mdr(imports, 'Description')
mdr_product

In [ ]:
# MDR by product — horizontal bar chart
colors = mdr_product['Risk'].map({'High': '#d62728', 'Moderate': '#ff7f0e', 'Low': '#2ca02c'})

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(mdr_product['Description'], mdr_product['MDR_pct'], color=colors)
ax.axvline(x=50, color='orange', linestyle='--', linewidth=1.5, label='50% threshold')
ax.set_xlabel('MDR (%)')
ax.set_title('Maritime Dependency Ratio by Product Category', fontsize=13, fontweight='bold')
ax.legend()

# Add value labels
for bar, val in zip(bars, mdr_product['MDR_pct']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

top3 = mdr_product.head(3)['Description'].tolist()
print(f'\n🔴 Highest risk products: {top3}')

In [ ]:
# MDR by supplier country — top 20
mdr_country = compute_mdr(imports, 'Area').head(20)

colors_c = mdr_country['Risk'].map({'High':'#d62728','Moderate':'#ff7f0e','Low':'#2ca02c'})

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(mdr_country['Area'], mdr_country['MDR_pct'], color=colors_c)
ax.axvline(x=50, color='orange', linestyle='--', linewidth=1.5)
ax.set_xlabel('MDR (%)')
ax.set_title('Top 20 Supplier Countries by Maritime Dependency', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# MDR trend over time by top-risk product categories
top_risk_cats = mdr_product[mdr_product['Risk'] == 'High']['Description'].tolist()

mdr_trend = []
for year, grp in imports[imports['Description'].isin(top_risk_cats)].groupby(imports['Year'].dt.year):
    for cat, sub in grp.groupby('Description'):
        sea = sub[sub['Transport'] == 'Sea']['FobValue'].sum()
        total = sub['FobValue'].sum()
        mdr_trend.append({'Year': year, 'Description': cat, 'MDR_pct': round(sea/total*100, 1)})

mdr_trend_df = pd.DataFrame(mdr_trend)

fig = px.line(mdr_trend_df, x='Year', y='MDR_pct', color='Description',
              markers=True, template='plotly_dark',
              title='MDR Over Time — High-Risk Product Categories',
              labels={'MDR_pct': 'MDR (%)', 'Description': 'Product'})
fig.add_hline(y=50, line_dash='dash', line_color='orange',
              annotation_text='50% threshold')
fig.show()

print('\n💡 Insight: Consistently high MDR across years = structural dependency, not a one-off.')

---
## 4. Herfindahl-Hirschman Index (HHI)

### What is HHI?

HHI measures how concentrated China's import supply is for each product category. It sums the squared market shares of all supplier countries:

$$\text{HHI} = \sum_{i=1}^{n} s_i^2 \times 10{,}000$$

where $s_i$ is country $i$'s share of total imports for that category.

| HHI Range | Risk Level | Interpretation |
|-----------|-----------|----------------|
| < 1,500   | 🟢 Low    | Supply spread across many countries |
| 1,500–2,500 | 🟡 Moderate | A few countries hold significant share |
| > 2,500   | 🔴 High   | 1–2 countries dominate — highly fragile |

In [ ]:
def compute_hhi(df: pd.DataFrame) -> pd.DataFrame:
    """Compute HHI per HS category per year."""
    g = df.groupby(['Year', 'Description', 'Area'])['FobValue'].sum().reset_index()
    totals = g.groupby(['Year', 'Description'])['FobValue'].sum().rename('Total').reset_index()
    g = g.merge(totals, on=['Year', 'Description'])
    g['share_sq'] = (g['FobValue'] / g['Total']) ** 2
    hhi = (g.groupby(['Year', 'Description'])['share_sq']
             .sum().reset_index())
    hhi['HHI'] = (hhi['share_sq'] * 10_000).round(0)
    hhi['RiskLevel'] = pd.cut(hhi['HHI'],
                               bins=[0, 1500, 2500, 10000],
                               labels=['Low Risk', 'Moderate Risk', 'High Risk'])
    return hhi[['Year', 'Description', 'HHI', 'RiskLevel']]


imports_yr = imports.copy()
imports_yr['Year'] = imports_yr['Year'].dt.year

hhi_df = compute_hhi(imports_yr)
hhi_df.head(10)

In [ ]:
# Latest year HHI snapshot
latest_year = hhi_df['Year'].max()
snapshot = (hhi_df[hhi_df['Year'] == latest_year]
            .sort_values('HHI', ascending=True))

risk_colors = {'Low Risk': '#2ca02c', 'Moderate Risk': '#ff7f0e', 'High Risk': '#d62728'}
bar_colors = snapshot['RiskLevel'].map(risk_colors)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(snapshot['Description'], snapshot['HHI'], color=bar_colors)
ax.axvline(x=1500, color='gold',   linestyle='--', linewidth=1.5, label='Moderate threshold (1500)')
ax.axvline(x=2500, color='tomato', linestyle='--', linewidth=1.5, label='High threshold (2500)')
ax.set_xlabel('HHI Score')
ax.set_title(f'Supplier Concentration (HHI) by Product — {latest_year}',
             fontsize=13, fontweight='bold')
ax.legend()

for bar, val, risk in zip(bars, snapshot['HHI'], snapshot['RiskLevel']):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{int(val):,}  ({risk})', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# HHI trend over time — line chart
fig = px.line(hhi_df, x='Year', y='HHI', color='Description',
              markers=True, template='plotly_dark',
              title='HHI Trend 2016–2021 by Product Category',
              labels={'HHI': 'HHI Score'})

fig.add_hrect(y0=0,    y1=1500,  fillcolor='green',  opacity=0.05,
              annotation_text='Low Risk',      annotation_position='right')
fig.add_hrect(y0=1500, y1=2500,  fillcolor='orange', opacity=0.05,
              annotation_text='Moderate Risk', annotation_position='right')
fig.add_hrect(y0=2500, y1=10000, fillcolor='red',    opacity=0.05,
              annotation_text='High Risk',     annotation_position='right')
fig.show()

---
## 5. Combined Risk Matrix

Plotting **MDR vs HHI** for each product creates a 2D risk matrix:
- **Top-right quadrant** = High MDR + High HHI = most dangerous (sea-dependent AND few suppliers)
- **Bottom-left** = Low MDR + Low HHI = most resilient

In [ ]:
# Merge MDR + HHI for latest year
hhi_latest = hhi_df[hhi_df['Year'] == latest_year][['Description', 'HHI', 'RiskLevel']]
combined = mdr_product.merge(hhi_latest, on='Description')

fig = px.scatter(
    combined,
    x='MDR_pct', y='HHI',
    text='Description',
    color='RiskLevel',
    size='MDR_pct',
    color_discrete_map={'Low Risk':'#2ca02c','Moderate Risk':'#ff7f0e','High Risk':'#d62728'},
    template='plotly_dark',
    title='Risk Matrix: Maritime Dependency vs Supplier Concentration',
    labels={'MDR_pct': 'Maritime Dependency Ratio (%)', 'HHI': 'HHI (Supplier Concentration)'}
)

fig.add_vline(x=50,   line_dash='dash', line_color='orange', opacity=0.5)
fig.add_hline(y=1500, line_dash='dash', line_color='gold',   opacity=0.5)
fig.update_traces(textposition='top center')
fig.show()

print('\n💡 Products in the top-right quadrant are highest priority for policy intervention.')

---
## 6. Key Findings

| # | Finding | Implication |
|---|---------|-------------|
| 1 | **Mineral Products** have the highest MDR (~75%+) | A Suez/South China Sea disruption would immediately hit energy imports |
| 2 | **Russia, Middle East, Africa** show highest country-level MDR | Geopolitical instability in these regions = direct supply risk |
| 3 | **Electrical Machinery** has High HHI — supply dominated by few countries | Single-source risk; COVID-19 showed how fragile this is |
| 4 | **MDR has been structurally stable** 2016–2021 | The dependency is not a short-term trend — it's built into trade routes |
| 5 | **High MDR + High HHI** products need immediate diversification | Both the transport mode AND supplier base need redundancy |

---
## 7. Recommendations

1. **Prioritize Mineral Products & Plastics** for supply chain resilience planning — highest MDR
2. **Diversify supplier base** for high-HHI categories — reduce single-country dependency
3. **Develop rail alternatives** along Belt & Road corridors to reduce sea reliance for inland routes
4. **Build strategic reserves** for top-right risk matrix products
5. **Monitor MDR annually** — if ratio increases, policy intervention is needed

---
*Notebook by Vigneshwari Nalla · Data: UNCTAD · China's Import Engine Project*